In [1]:
import time
from pathlib import Path
import pandas as pd
from nba_api.stats.endpoints import leaguedashplayerstats

DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

SEASONS = [f"{y}-{str(y+1)[-2:]}" for y in range(2013, 2026)]

BASE_COLS = ["PLAYER_ID", "SEASON", "PLAYER_NAME", "TEAM_ID", "TEAM_COUNT",
             "AGE", "GP", "MIN", "PTS", "FGA", "FG_PCT", "FG3A", "FG3_PCT",
             "FTA", "FT_PCT"]
ADV_COLS  = ["PLAYER_ID", "SEASON", "USG_PCT", "TS_PCT", "PACE"]

In [2]:
DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

def get_season_stats(season, measure_type="Base"):
    path = DATA_DIR / f"{measure_type.lower()}_{season}.csv"
    if path.exists():
        return pd.read_csv(path)

    resp = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        season_type_all_star="Regular Season",
        per_mode_detailed="PerGame",
        measure_type_detailed_defense=measure_type,
        timeout=60,
    )
    df = resp.get_data_frames()[0]
    df["SEASON"] = season
    df.to_csv(path, index=False)
    time.sleep(2)
    return df

In [3]:
for s in SEASONS:
    get_season_stats(s, "Base")
    get_season_stats(s, "Advanced")

In [4]:
frames = []
for s in SEASONS:
    base = pd.read_csv(DATA_DIR / f"base_{s}.csv")[BASE_COLS]
    adv  = pd.read_csv(DATA_DIR / f"advanced_{s}.csv")[ADV_COLS]
    merged = base.merge(adv, on=["PLAYER_ID", "SEASON"],
                        how="inner", validate="one_to_one")
    frames.append(merged)

df = pd.concat(frames, ignore_index=True)

In [5]:
print(df.shape)
print(df.duplicated(["PLAYER_ID", "SEASON"]).sum())      # debe ser 0
print(df.isna().sum()[lambda x: x > 0])                  # columnas con nulos
print(df.groupby("SEASON").size())                       # jugadores por temporada
print(df[["GP", "MIN", "PTS"]].describe())

(6942, 18)
0
Series([], dtype: int64)
SEASON
2013-14    482
2014-15    492
2015-16    476
2016-17    486
2017-18    540
2018-19    530
2019-20    529
2020-21    540
2021-22    605
2022-23    539
2023-24    572
2024-25    569
2025-26    582
dtype: int64
                GP          MIN          PTS
count  6942.000000  6942.000000  6942.000000
mean     47.934601    19.557591     8.569317
std      25.075039     9.386572     6.259447
min       1.000000     0.400000     0.000000
25%      27.000000    12.100000     3.900000
50%      54.000000    19.200000     7.000000
75%      69.000000    27.400000    11.800000
max      84.000000    43.500000    36.100000


In [6]:
Path("data/processed").mkdir(parents=True, exist_ok=True)
df.to_csv("data/processed/player_seasons_raw.csv", index=False)